# 11주차. 몬테카를로 시뮬레이션

## 오늘의 목표

이번 시간에는 통계 이론을 배우지 않는다. **불확실한 입력값을 분포로 표현하고 1만 번 굴려서 결과 분포를 얻는** 사고방식을 익힌다.

수업이 끝나면 다음을 할 수 있어야 한다.

1. 점 추정과 확률적 추정의 차이를 한 문장으로 설명한다.
2. 정규·삼각·균등·로그정규 분포를 골라서 쓴다.
3. 매출·비용·이익을 분포로 표현해 시뮬레이션한다.
4. 결과 분포를 평균·90% 구간·손실 확률로 해석한다.
5. 토네이도 차트로 핵심 리스크 변수를 식별한다.
6. 시나리오 플래닝과 시뮬레이션을 통합한다.

## 1. 점 추정 vs 분포 추정

기획자에게 5년 누적 순이익을 물으면 흔히 한 숫자로 답한다.

> 50억

정직한 답은 분포다.

> 평균 50억, 90% 구간 30억~70억, 손실 확률 8%

이 답이 의사결정에 훨씬 유용하다. 임원이 묻는 질문은 "평균이 얼마인가"가 아니라 "최악의 경우는 어디인가"이기 때문이다.

**한 줄 요약**

> 점 추정은 "가장 그럴듯한 한 값"을 답하고, 시뮬레이션은 "가능한 모든 값의 빈도"를 답한다.

## 2. 분포 4종 매뉴얼

어떤 변수에 어떤 분포를 쓸지 외울 필요는 없다. 이 표만 보고 고르면 된다.

| 분포 | 언제 쓰는가 | 입력 |
|------|-----------|------|
| **정규(Normal)** | 평균 주변 대칭 (측정 오차, 평소 매출) | 평균, 표준편차 |
| **삼각(Triangular)** | 최소·최빈·최대 세 값을 알 때 (전문가 추정) | min, mode, max |
| **균등(Uniform)** | 범위만 아는 무지 상태 (거시 충격 ±5%) | min, max |
| **로그정규(Lognormal)** | 양수, 우측 꼬리 (사고 비용, 대박 매출) | 로그 평균, 로그 표준편차 |

팁:

- 잘 모르겠으면 **삼각 분포**가 가장 안전하다 (전문가 의견 그대로 반영).
- "가끔 큰 값이 튄다"는 변수는 **로그정규**.
- 정규를 남용하지 말 것 — 손실 비용 같은 양수 변수에는 부적절.

In [ ]:
# 기본 라이브러리 불러오기
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# 4가지 분포 비교 시각화
rng = np.random.default_rng(42)
samples = {
    "Normal(10, 2)":          rng.normal(10, 2, 5000),
    "Triangular(5, 10, 20)":  rng.triangular(5, 10, 20, 5000),
    "Uniform(5, 15)":         rng.uniform(5, 15, 5000),
    "Lognormal(2.2, 0.4)":    rng.lognormal(2.2, 0.4, 5000),
}

fig, axes = plt.subplots(2, 2, figsize=(9, 6))
for ax, (name, s) in zip(axes.ravel(), samples.items()):
    ax.hist(s, bins=40, color="white", edgecolor="black", linewidth=0.6)
    ax.set_title(name, color="black")
    ax.set_xlabel("value", color="black")
    ax.set_ylabel("count", color="black")
plt.tight_layout()
plt.show()

## 3. 시뮬레이션 매뉴얼 (6단계)

1. **결과 변수**를 정한다 (예: 5년 누적 순이익).
2. **입력 변수**를 모두 적는다 (매출, 변동비율, 고정비, 거시 충격).
3. 각 입력에 **분포와 파라미터**를 정한다 (전문가 의견 또는 데이터).
4. **N=10,000번** 시뮬레이션한다.
5. 결과 분포의 **평균·5%·95% 분위수·손실 확률**을 본다.
6. **토네이도 차트**로 어느 입력이 결과를 가장 흔드는지 본다.

사례: **청년 일자리 정책의 5년 누적 순이익**

- 연 매출 = `Normal(10, 2)` 억 원
- 변동비율 = `Triangular(0.4, 0.5, 0.7)`
- 연 고정비 = `Normal(3, 0.5)` 억 원
- 거시 충격 = `Uniform(-0.05, 0.05)` (매출에 곱)

In [ ]:
n = 10_000
rng = np.random.default_rng(42)

annual_revenue       = rng.normal(10.0, 2.0, n)
variable_cost_ratio  = rng.triangular(0.4, 0.5, 0.7, n)
annual_fixed         = rng.normal(3.0, 0.5, n)
macro_shock          = rng.uniform(-0.05, 0.05, n)

annual_profit = annual_revenue * (1 - variable_cost_ratio) * (1 + macro_shock) - annual_fixed
five_year_profit = 5 * annual_profit

results = pd.DataFrame({
    "annual_revenue": annual_revenue,
    "variable_cost_ratio": variable_cost_ratio,
    "annual_fixed": annual_fixed,
    "macro_shock": macro_shock,
    "five_year_profit": five_year_profit,
})
results.head()

## 4. 결과 분포 해석

히스토그램이 분포의 모양을 보여준다. 0보다 작은 영역의 비율이 손실 확률이다.

**판정 매뉴얼**

| 손실 확률 P(이익 < 0) | 판정 |
|---------------------|------|
| < 5% | Go (안전) |
| 5~20% | Hold (리스크 완화 후 재판단) |
| > 20% | Stop or redesign |

In [ ]:
summary = {
    "mean":      round(five_year_profit.mean(), 2),
    "p05":       round(np.percentile(five_year_profit, 5), 2),
    "p95":       round(np.percentile(five_year_profit, 95), 2),
    "P(loss<0)": round((five_year_profit < 0).mean(), 3),
}
print(summary)

plt.figure(figsize=(7, 4))
plt.hist(five_year_profit, bins=40, color="white", edgecolor="black", linewidth=0.6)
plt.axvline(0, color="black", linewidth=1)
plt.axvline(summary["mean"], color="gray", linestyle="--", linewidth=0.8)
plt.xlabel("5-year profit (억 원)", color="black")
plt.ylabel("count", color="black")
plt.title("5-Year Profit Distribution (N=10,000)", color="black")
plt.tight_layout()
plt.show()

## 5. 토네이도 차트: 어느 입력이 가장 중요한가

각 입력 변수를 **±1 표준편차**로 흔들었을 때 평균 결과가 얼마나 변하는지를 막대로 그린다. 막대가 길수록 그 변수가 **핵심 리스크**다.

**기획 적용**

- 1위 변수에 추가 조사·시범사업·계약 보호장치를 집중한다.
- 끝 순위 변수는 평균값으로 고정해도 된다 (시뮬 단순화).

In [ ]:
inputs = ["annual_revenue", "variable_cost_ratio", "annual_fixed", "macro_shock"]
rows = []
for col in inputs:
    median = results[col].median()
    std = results[col].std()
    low_mean = results.loc[results[col] < (median - std), "five_year_profit"].mean()
    high_mean = results.loc[results[col] > (median + std), "five_year_profit"].mean()
    rows.append({"variable": col, "low_mean": low_mean, "high_mean": high_mean,
                 "swing": abs(high_mean - low_mean)})
tornado_df = pd.DataFrame(rows).sort_values("swing", ascending=False)

plt.figure(figsize=(7, 4))
plt.barh(tornado_df["variable"], tornado_df["swing"], color="white", edgecolor="black")
plt.xlabel("Swing of mean profit", color="black")
plt.title("Tornado Chart", color="black")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

tornado_df.round(2)

## 6. 시나리오 × 시뮬레이션 통합

10주차에서 만든 4개 시나리오마다 입력값 분포를 다르게 잡고 시뮬레이션한다. 같은 사업이 시나리오마다 어떻게 다른 결과를 내는지 본다.

이번 예시에서는 시나리오마다 **매출 평균과 변동비 최빈값**만 바꾼다.

In [ ]:
rng = np.random.default_rng(0)
scenarios = {
    "Green Sprint":     {"revenue_mean": 12.0, "cost_mode": 0.45},
    "Tech Race":        {"revenue_mean": 11.0, "cost_mode": 0.50},
    "Stuck in Traffic": {"revenue_mean":  9.0, "cost_mode": 0.55},
    "Slow Transition":  {"revenue_mean":  8.0, "cost_mode": 0.60},
}

rows = []
for name, p in scenarios.items():
    revenue = rng.normal(p["revenue_mean"], 2.0, n)
    cost_ratio = rng.triangular(0.4, p["cost_mode"], 0.7, n)
    fixed = rng.normal(3.0, 0.5, n)
    profit = 5 * (revenue * (1 - cost_ratio) - fixed)
    rows.append({
        "scenario": name,
        "mean": round(profit.mean(), 2),
        "p05": round(np.percentile(profit, 5), 2),
        "p95": round(np.percentile(profit, 95), 2),
        "P(loss<0)": round((profit < 0).mean(), 3),
    })

scenario_table = pd.DataFrame(rows)
scenario_table

**해석 매뉴얼**

- 시나리오별 손실 확률이 모두 낮으면 **강건한 사업**이다.
- 어느 시나리오에서 손실 확률이 30%를 넘으면, 그 시나리오가 현실이 되었을 때의 **헤지 전략**(보험·계약 조항·축소 가능 구조)을 미리 설계한다.
- 가장 좋은 시나리오 결과가 평균보다 너무 크면 **베이스 케이스를 과소평가**했을 가능성이 있다.

## 7. 바이브 코딩 프롬프트

오늘 실습에서 AI에게 요청할 문장이다.

**1단계: 입력 변수와 분포 추천**

```text
나는 [정책/사업명]의 [결과 변수, 예: 5년 누적 순이익]을 시뮬레이션하려고 해.
관련 입력 변수 4~6개를 추천해주고, 각각에 적절한 분포(정규/삼각/균등/로그정규)와 파라미터를 알려줘.
표로 정리해줘.
```

**2단계: 시뮬레이션 코드**

```text
위 분포 설정으로 N=10,000번 시뮬레이션하는 Python 코드를 만들어줘.
결과 분포 히스토그램(흑백)과 평균·5%·95%·손실 확률을 출력해줘.
```

**3단계: 토네이도**

```text
각 입력 변수를 ±1 표준편차로 흔들었을 때 결과 평균이 얼마나 변하는지 토네이도 차트로 그려줘.
막대가 긴 순서대로 정렬해줘.
```

**4단계: 시나리오 통합**

```text
10주차 4개 시나리오마다 입력 평균을 다르게 잡고, 각 시나리오 결과 분포의 평균·5%·95%·손실 확률을 비교 표로 정리해줘.
어떤 시나리오에서 가장 위험한지도 한 줄로 알려줘.
```

## 8. 학생 실습

관심 있는 정책 또는 사업을 하나 골라 시뮬레이션을 진행한다.

예시 주제:

- 청년 창업 지원금의 5년 누적 효과
- 신제품 출시 1년차 영업이익
- 정책 시범사업의 비용·편익 비
- 학교 급식 예산 변동성

**단계**

1. 결과 변수를 정한다.
2. 입력 변수 4~6개를 적는다.
3. 각 입력의 분포와 파라미터를 정한다 (AI 추천 → 본인 검증).
4. N=10,000번 시뮬레이션한다.
5. 평균·5%·95%·손실 확률을 적는다.
6. 토네이도 차트로 핵심 리스크를 식별한다.
7. 결정 권고를 작성한다.

In [ ]:
# 여기에 AI가 만들어준 시뮬레이션 코드를 붙여 넣으세요.

my_n = 10_000
my_rng = np.random.default_rng(2026)

# 예시:
# my_input1 = my_rng.normal(평균, 표준편차, my_n)
# my_input2 = my_rng.triangular(min, mode, max, my_n)
# my_result = ...
# print(round(my_result.mean(), 2), round(np.percentile(my_result, 5), 2), round((my_result < 0).mean(), 3))

## 9. 제출물

아래 표를 작성하여 제출한다.

| 항목 | 작성 내용 |
|------|----------|
| 결과 변수 | |
| 입력 변수와 분포 (3개 이상) | |
| 평균 결과 | |
| 5% / 95% 분위수 | |
| 손실 확률 P(결과 < 0) | |
| 핵심 리스크 변수 (토네이도 1위) | |
| 의사결정 권고 (Go / Hold / Stop) | |
| 시나리오별 결과 비교 (선택) | |

## 오늘의 핵심

몬테카를로는 "가장 그럴듯한 답 하나"를 찾는 도구가 아니다. **가능한 결과의 분포를 보여주고, 최악·평균·최선이 어디인지를 정량적으로 답하는** 도구다. 의사결정자는 평균이 아니라 **꼬리(tail)** 를 본다.